In [2]:
from treys import Card, Evaluator, Deck
import random
import itertools

In [3]:
class RangeParser:
    SUITS = ['s', 'h', 'd', 'c']
    RANKS = '23456789TJQKA'
    
    @staticmethod
    def get_all_combos(rank1, rank2, suited=None):
        combos = []
        # Case 1: Pairs (e.g., AA)
        if rank1 == rank2:
            for s1, s2 in itertools.combinations(RangeParser.SUITS, 2):
                combos.append([rank1 + s1, rank2 + s2])
        # Case 2: Suited (e.g., AKs)
        elif suited is True:
            for s in RangeParser.SUITS:
                combos.append([rank1 + s, rank2 + s])
        # Case 3: Offsuit (e.g., AKo)
        elif suited is False:
            for s1 in RangeParser.SUITS:
                for s2 in RangeParser.SUITS:
                    if s1 != s2:
                        combos.append([rank1 + s1, rank2 + s2])
        # Case 4: Any (Both Suited and Offsuit)
        else:
            for s1 in RangeParser.SUITS:
                for s2 in RangeParser.SUITS:
                    if s1 == s2 or s1 != s2: # All 16 combos
                        if rank1 + s1 != rank2 + s2:
                            # Avoid duplicates like [As, Ks] and [Ks, As]
                            hand = sorted([rank1 + s1, rank2 + s2], key=lambda x: RangeParser.RANKS.find(x[0]), reverse=True)
                            if hand not in combos:
                                combos.append(hand)
        return combos

    def parse_range(self, range_str):
        """ Parse -> "AA, 88+, AKs, JTs-QJs" """
        final_range = []
        parts = [p.strip() for p in range_str.split(',')]
        
        for part in parts:
            if '+' in part:
                # Process 88+, AKs+, e.g. range
                base = part.replace('+', '')
                r1 = base[0]
                r2 = base[1]
                type_suffix = base[2:] if len(base) > 2 else ""
                
                idx1 = self.RANKS.find(r1)
                idx2 = self.RANKS.find(r2)
                
                if idx1 == idx2: # Pairs (88+)
                    for i in range(idx1, len(self.RANKS)):
                        final_range.extend(self.get_all_combos(self.RANKS[i], self.RANKS[i]))
                else: # Non-pairs (AJs+)
                    for i in range(idx2, idx1):
                        suited = True if 's' in type_suffix else (False if 'o' in type_suffix else None)
                        final_range.extend(self.get_all_combos(r1, self.RANKS[i], suited))
            
            elif '-' in part:
                # Process (-) 89o-JTo
                pass
            
            else:
                # Single Combo AKs, 7h8h, QQ
                if len(part) == 4: # Specific hand like 7h8h
                    final_range.append([part[:2], part[2:]])
                elif len(part) == 3: # AKs, AKo
                    final_range.extend(self.get_all_combos(part[0], part[1], part[2] == 's'))
                elif len(part) == 2: # QQ
                    final_range.extend(self.get_all_combos(part[0], part[1]))
                    
        return final_range

def range_vs_range_equity(hero_hand_strs, villain_range_strs, board_strs, n_simulations=10000):
    evaluator = Evaluator()
    
    # 1. (['As', 'Ks'] -> [Card.new('As'), Card.new('Ks')])
    hero_hand = [Card.new(c) for c in hero_hand_strs]
    board = [Card.new(c) for c in board_strs]
    
    villain_range = [[Card.new(c1), Card.new(c2)] for c1, c2 in villain_range_strs]
    
    hero_wins = 0
    ties = 0
    
    #Simulation
    for _ in range(n_simulations):
        # Random Range from villain
        villain_hand = random.choice(villain_range)
        
        # Check (Dead Card Check)
        all_dead_cards = set(hero_hand + board + villain_hand)
        if len(all_dead_cards) < (len(hero_hand) + len(board) + len(villain_hand)):
            continue
            
        # Board (Turn and River)
        full_deck = Deck()
        remaining_cards = [c for c in full_deck.cards if c not in all_dead_cards]
        needed = 5 - len(board)
        sim_board = board + random.sample(remaining_cards, needed)
        
        # Evalueation
        hero_score = evaluator.evaluate(hero_hand, sim_board)
        villain_score = evaluator.evaluate(villain_hand, sim_board)
        
        if hero_score < villain_score:
            hero_wins += 1
        elif hero_score == villain_score:
            ties += 1
            
    equity = (hero_wins + (ties * 0.5)) / n_simulations
    return equity 
    
def compute_call_ev(equity, pot, call_cost, future_cost=0):

    return equity * (pot + call_cost) - call_cost - future_cost

def compute_fold_ev():

    return 0

def breakeven_equity(pot, call_cost):
    Breakeven_Eq = call_cost / (pot + call_cost)

    return f"Break Even Equity: {Breakeven_Eq:.2%}"

def decide_call_or_fold(equity, pot, call_cost):

    call_ev = compute_call_ev(equity, pot, call_cost)
    fold_ev = compute_fold_ev()
    decision = "CALL" if call_ev > fold_ev else "FOLD"

    return f"equity: {equity:.2%}, call_ev: {call_ev:.4f}, fold_ev: {fold_ev}, decision: {decision}" 

parser = RangeParser() #
hero = ['Jh', 'Qs']
villain_r =  parser.parse_range("8hTd")
flop = ['Jd', '9h', 'Ts']

equity = range_vs_range_equity(hero, villain_r, flop)
print(f"Hero Hands: {hero}, Hero Equity: {equity:.2%}")

pot = 100
call_cost = 50
result = decide_call_or_fold(equity, pot, call_cost)
print(result)
print(breakeven_equity(pot, call_cost))

Hero Hands: ['Jh', 'Qs'], Hero Equity: 69.04%
equity: 69.04%, call_ev: 53.5600, fold_ev: 0, decision: CALL
Break Even Equity: 33.33%
